# Planned InstanSeg seed-threshold sweep (compact fixed crop)

The active history has no genuine same-image, multi-threshold WSI/resolver sweep. The prior mask-threshold sweep changes mask_threshold; the reconciliation notebook only probes continuous peak maps; and the recent seed-0.1 notebook is one end-to-end pass. This notebook prepares thresholds 0.05, 0.10, 0.20, 0.40, and 0.60 with all other M11 v4 settings fixed.

It derives one 4096 x 4096 native-pixel crop from all 72 channels of the existing SLIDE-0330 half crop, including the prior hotspot y=24748:25172, x=13818:14243. It is prepared but not run here. Use the instanseg_nimbus kernel, the patched editable InstanSeg fork, and set RUN_SWEEP=True only for GPU execution.


In [ ]:
from pathlib import Path
import json, subprocess, sys, time, xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import numpy as np, pandas as pd, tifffile, yaml, zarr
from skimage.segmentation import find_boundaries

MIF_PIPELINE_ROOT=Path("/data1/lowes/ratnayn/Codex/projects/mIF-pipeline")
INSTANSEG_ROOT=Path("/data1/lowes/ratnayn/Codex/projects/instanseg")
SOURCE_HALF_CROP=Path("/data1/lowes/ratnayn/Codex/codex-scratch/mIF-pipeline/instanseg_watershed_production_smoke_all_channel_crop/SLIDE-0330/SLIDE-0330_all_channels_half_crop.ome.tif")
M11_CONFIG=Path("/data1/lowes/ratnayn/Analysis/M11_guidepool/M11_slides_config_v4.yaml")
CROP_BOUNDS_YX=(22528,26624,11776,15872); CROP_H=CROP_W=4096
HOTSPOT_SOURCE_BOUNDS=(24748,25172,13818,14243)
HOTSPOT_CROP_BOUNDS=(2220,2644,2042,2467); FIXED_NATIVE_BOUNDS=(2124,2740,1946,2563)
SEED_THRESHOLDS=(0.05,0.10,0.20,0.40,0.60)
MODEL_NAME="fluorescence_nuclei_and_cells"; PIXEL_SIZE_UM=0.325; REFERENCE_CHANNEL="R1_DAPI"
NORMALIZATION_PERCENTILES=(0.1,99.9); WSI_TILE_SIZE=2048; WSI_OVERLAP=80; WSI_DETECTION_SIZE=20; WSI_BATCH_SIZE=1
CROP_ROOT=SOURCE_HALF_CROP.parent/"seed_threshold_sweep_crop"; SMALL_CROP=CROP_ROOT/"SLIDE-0330_seed_threshold_sweep_4096.ome.tif"
BASE_OUTPUT_DIR=CROP_ROOT/"results"; SUMMARY_CSV=BASE_OUTPUT_DIR/"seed_threshold_sweep_summary.csv"; SUMMARY_JSON=BASE_OUTPUT_DIR/"seed_threshold_sweep_summary.json"; COMPARISON_PNG=BASE_OUTPUT_DIR/"seed_threshold_sweep_fixed_hotspot.png"
GENERATE_SMALL_CROP=True; CROP_OVERWRITE=False; RUN_SWEEP=False; REUSE_COMPLETED_OUTPUTS=True; RUN_CLEANUP=True; CLEANUP_REUSE=True; CLEANUP_OVERWRITE=False
CROP_ROOT.mkdir(parents=True,exist_ok=True); BASE_OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

def ome_parts(path):
    with tifffile.TiffFile(str(path)) as tif:
        series=tif.series[0]; xml=tif.ome_metadata or ""; out={"shape":tuple(map(int,series.shape)),"axes":str(series.axes),"dtype":np.dtype(series.dtype)}
    root=ET.fromstring(xml); px=next(e for e in root.iter() if e.tag.rsplit("}",1)[-1]=="Pixels")
    out["channels"]=tuple(e.attrib["Name"] for e in px if e.tag.rsplit("}",1)[-1]=="Channel"); out["physical_x"]=float(px.attrib["PhysicalSizeX"]); out["physical_y"]=float(px.attrib.get("PhysicalSizeY",out["physical_x"])); return out

if not SOURCE_HALF_CROP.is_file() or not M11_CONFIG.is_file(): raise FileNotFoundError("source crop or M11 config missing")
m11=yaml.safe_load(M11_CONFIG.read_text()); mi=dict(m11["instanseg"])
assert mi["model"]==MODEL_NAME and mi["mode"]=="wsi_global" and float(m11["pixel_size_um"])==PIXEL_SIZE_UM
assert tuple(mi["normalization_percentiles"])==NORMALIZATION_PERCENTILES and mi["reference_channel"]==REFERENCE_CHANNEL
for k,v in (("tile_size",WSI_TILE_SIZE),("overlap",WSI_OVERLAP),("detection_size",WSI_DETECTION_SIZE),("batch_size",WSI_BATCH_SIZE)): assert int(mi[k])==v
assert mi["resolution_method"]=="watershed" and bool(mi["allow_unnucleated_cells"]) and bool(mi["cleanup_fragments"])
source_meta=ome_parts(SOURCE_HALF_CROP)
assert source_meta["shape"]==(72,27680,31344) and source_meta["axes"]=="CYX" and source_meta["dtype"]==np.dtype("uint16")
assert len(source_meta["channels"])==72 and abs(source_meta["physical_x"]-PIXEL_SIZE_UM)<1e-3
M11_CHANNELS=tuple(mi["channels"]); CHANNEL_IDS=[source_meta["channels"].index(x) for x in M11_CHANNELS]; REFERENCE_CHANNEL_ID=source_meta["channels"].index(REFERENCE_CHANNEL)
if str(INSTANSEG_ROOT) not in sys.path: sys.path.insert(0,str(INSTANSEG_ROOT))
import instanseg
from instanseg import InstanSeg
from tiffslide import TiffSlide
import instanseg.inference_class as inference_class
inference_class.TiffSlide=TiffSlide
FORK_COMMIT=subprocess.check_output(["git","rev-parse","HEAD"],cwd=INSTANSEG_ROOT,text=True).strip()
print({"source":str(SOURCE_HALF_CROP),"crop_bounds_yx":CROP_BOUNDS_YX,"hotspot_crop_bounds":HOTSPOT_CROP_BOUNDS,"m11_channels":list(M11_CHANNELS),"channel_ids":CHANNEL_IDS,"fork_commit":FORK_COMMIT,"run_sweep":RUN_SWEEP})


## 1. Generate and validate the all-channel 4096 x 4096 crop

The writer preserves source CYX order, all 72 names, uint16 dtype, tiled BigTIFF/OME metadata, and physical pixel size. It validates shape and pixel identity at multiple sampled blocks.


In [ ]:
def write_crop():
    if SMALL_CROP.exists() and not CROP_OVERWRITE: return print("STATUS: validating existing crop",SMALL_CROP)
    if SMALL_CROP.exists(): raise RuntimeError("Overwrite disabled; choose another isolated output")
    partial=SMALL_CROP.with_name(SMALL_CROP.name+".partial")
    if partial.exists(): raise FileExistsError(partial)
    ome=tifffile.OmeXml(); ome.addimage(np.dtype("uint16"),(72,CROP_H,CROP_W),(72,1,1,CROP_H,CROP_W,1),axes="CYX",Channel={"Name":list(source_meta["channels"])},PhysicalSizeX=source_meta["physical_x"],PhysicalSizeY=source_meta["physical_y"])
    print("STATUS: writing 72 channels")
    with tifffile.TiffFile(str(SOURCE_HALF_CROP)) as tif:
        store=tif.series[0].aszarr(level=0); src=zarr.open(store,mode="r")
        try:
            with tifffile.TiffWriter(str(partial),bigtiff=True,ome=False) as writer:
                for c,name in enumerate(source_meta["channels"]):
                    print(f"STATUS: channel {c+1}/72 {name}")
                    plane=np.asarray(src[c,CROP_BOUNDS_YX[0]:CROP_BOUNDS_YX[1],CROP_BOUNDS_YX[2]:CROP_BOUNDS_YX[3]],dtype=np.uint16)
                    assert plane.shape==(CROP_H,CROP_W)
                    writer.write(plane,compression="zlib",photometric="minisblack",tile=(512,512),metadata=None,description=ome.tostring() if c==0 else None)
        finally: store.close()
    partial.replace(SMALL_CROP); print("STATUS: crop written",SMALL_CROP)

def validate_crop():
    meta=ome_parts(SMALL_CROP)
    assert meta["shape"]==(72,CROP_H,CROP_W) and meta["axes"]=="CYX" and meta["dtype"]==np.dtype("uint16") and meta["channels"]==source_meta["channels"]
    assert abs(meta["physical_x"]-source_meta["physical_x"])<1e-9 and abs(meta["physical_y"]-source_meta["physical_y"])<1e-9
    with tifffile.TiffFile(str(SOURCE_HALF_CROP)) as st, tifffile.TiffFile(str(SMALL_CROP)) as ct:
        ss,cs=st.series[0].aszarr(level=0),ct.series[0].aszarr(level=0); src,out=zarr.open(ss,mode="r"),zarr.open(cs,mode="r")
        try:
            for y,x in ((0,0),(1024,1536),(CROP_H-64,CROP_W-64)):
                a=np.asarray(src[:,CROP_BOUNDS_YX[0]+y:CROP_BOUNDS_YX[0]+y+64,CROP_BOUNDS_YX[2]+x:CROP_BOUNDS_YX[2]+x+64]); b=np.asarray(out[:,y:y+64,x:x+64]); assert np.array_equal(a,b),(y,x); print("STATUS: pixel identity passed",y,x)
        finally: cs.close(); ss.close()
    print({"shape":meta["shape"],"axes":meta["axes"],"dtype":str(meta["dtype"]),"channels":len(meta["channels"]),"physical_um":(meta["physical_x"],meta["physical_y"]),"bounds_yx":CROP_BOUNDS_YX})

if GENERATE_SMALL_CROP and not SMALL_CROP.exists(): write_crop()
if not SMALL_CROP.is_file(): raise FileNotFoundError(SMALL_CROP)
validate_crop()


## 2. GPU-gated inference, cleanup, provenance, and comparison

Each threshold is isolated under results/seed_threshold_*. Incompatible or partial outputs raise instead of being overwritten. Cleanup uses the existing provisional 8-connected helper and writes separate Zarr/CSV/JSON/overview artifacts.


In [ ]:
helper_dir=MIF_PIPELINE_ROOT/"notebooks"
if str(helper_dir) not in sys.path: sys.path.insert(0,str(helper_dir))
from instanseg_connectedness_cleanup import run_cleanup, INSTANSEG_MIN_SIZE

def tok(t): return f"{float(t):.2f}".replace(".","p")
def paths(t):
    d=BASE_OUTPUT_DIR/f"seed_threshold_{tok(t)}"
    return {"dir":d,"resolved":d/f"SLIDE-0330_watershed_resolved_seed_threshold_{tok(t)}.zarr","cleaned":d/f"SLIDE-0330_watershed_resolved_seed_threshold_{tok(t)}_postresolution_8conn_min10.zarr","csv":d/f"SLIDE-0330_seed_threshold_{tok(t)}_cleanup_per_id.csv","json":d/f"SLIDE-0330_seed_threshold_{tok(t)}_cleanup_summary.json","overview":d/f"SLIDE-0330_seed_threshold_{tok(t)}_removed_overview.png"}
def prov(t):
    return {"experiment":"seed_threshold_sweep_small_crop","source_image":str(SMALL_CROP.resolve()),"source_shape":[CROP_H,CROP_W],"crop_bounds_yx":list(CROP_BOUNDS_YX),"channels":list(M11_CHANNELS),"channel_ids":list(CHANNEL_IDS),"reference_channel":REFERENCE_CHANNEL,"reference_channel_id":REFERENCE_CHANNEL_ID,"model":MODEL_NAME,"pixel_size_um":PIXEL_SIZE_UM,"normalization_percentiles":list(NORMALIZATION_PERCENTILES),"tile_size":WSI_TILE_SIZE,"overlap":WSI_OVERLAP,"detection_size":WSI_DETECTION_SIZE,"batch_size":WSI_BATCH_SIZE,"resolve_cell_and_nucleus":True,"resolution_method":"watershed","allow_unnucleated_cells":True,"cleanup_fragments":True,"seed_threshold":float(t),"instanseg_fork":str(INSTANSEG_ROOT.resolve()),"instanseg_fork_commit":FORK_COMMIT}
def compatible(q,t):
    if not Path(q).is_dir(): return False
    try:
        a=zarr.open(str(q),mode="r"); x=dict(a.attrs); p=dict(x.get("experiment_provenance") or {}); s=dict(x.get("wsi_settings") or {}); r=dict(x.get("resolution") or {})
        return x.get("status")=="complete" and tuple(a.shape[:1])==(2,) and x.get("planes")==["nuclei","cells"] and x.get("source_image")==str(SMALL_CROP.resolve()) and list(x.get("channel_ids",[]))==CHANNEL_IDS and p==prov(t) and s.get("tile_size")==WSI_TILE_SIZE and s.get("overlap")==WSI_OVERLAP and s.get("detection_size")==WSI_DETECTION_SIZE and s.get("batch_size")==WSI_BATCH_SIZE and s.get("resolve_cell_and_nucleus") is True and s.get("resolution_method")=="watershed" and s.get("seed_threshold")==float(t) and r.get("method")=="watershed" and r.get("allow_unnucleated_cells") is True and r.get("seed_threshold")==float(t)
    except Exception: return False
def stamp(q,t):
    a=zarr.open(str(q),mode="r+"); s=dict(a.attrs.get("wsi_settings") or {}); s.update({"tile_size":WSI_TILE_SIZE,"overlap":WSI_OVERLAP,"detection_size":WSI_DETECTION_SIZE,"batch_size":WSI_BATCH_SIZE,"resolve_cell_and_nucleus":True,"resolution_method":"watershed","seed_threshold":float(t)}); a.attrs["wsi_settings"]=s; r=dict(a.attrs.get("resolution") or {}); r.update({"method":"watershed","allow_unnucleated_cells":True,"seed_threshold":float(t)}); a.attrs["resolution"]=r; a.attrs["experiment_provenance"]=prov(t)
def labels(q):
    a=zarr.open(str(q),mode="r"); result=[]; pixels=[]
    for plane in range(2):
        ids=set(); npx=0
        for y in range(0,a.shape[1],a.chunks[1]):
            for x in range(0,a.shape[2],a.chunks[2]):
                b=np.asarray(a[plane,y:min(y+a.chunks[1],a.shape[1]),x:min(x+a.chunks[2],a.shape[2])]); v,n=np.unique(b,return_counts=True); k=v>0; ids.update(int(i) for i in v[k]); npx+=int(n[k].sum())
        result.append(ids); pixels.append(npx)
    return result,pixels
def attrs_summary(q):
    a=dict(zarr.open(str(q),mode="r").attrs); v=dict(a.get("validation") or {}); r=dict(a.get("resolution_summary") or {}); n=dict(a.get("normalization") or {})
    keys=("method","percentiles","bounds","channel_ids","reference_channel_id","source_shape","source_dtype","channel_pixel_counts")
    return {"raw_nuclei":v.get("raw_nuclei"),"raw_cells":v.get("raw_cells"),"proxy_cells":v.get("proxy_cells",r.get("unmatched_nuclei")),"unmatched_nuclei":v.get("unmatched_nuclei",r.get("unmatched_nuclei")),"normalization_signature":{k:n.get(k) for k in keys}}
def run_one(model,t):
    q=paths(t); q["dir"].mkdir(parents=True,exist_ok=True)
    if compatible(q["resolved"],t) and REUSE_COMPLETED_OUTPUTS: print("STATUS: reuse",t); return q,"reused",None
    if q["resolved"].exists(): raise FileExistsError(f"incompatible or partial output: {q['resolved']}")
    start=time.perf_counter(); print("STATUS: GPU inference",t)
    out=model.eval_whole_slide_image_global_normalization(str(SMALL_CROP),channel_ids=CHANNEL_IDS,pixel_size=PIXEL_SIZE_UM,normalization_percentiles=NORMALIZATION_PERCENTILES,reference_channel_id=REFERENCE_CHANNEL_ID,tile_size=WSI_TILE_SIZE,overlap=WSI_OVERLAP,detection_size=WSI_DETECTION_SIZE,batch_size=WSI_BATCH_SIZE,output_path=q["resolved"],overwrite=False,resolve_cell_and_nucleus=True,resolution_method="watershed",allow_unnucleated_cells=True,cleanup_fragments=True,seed_threshold=float(t))
    assert Path(out).resolve()==q["resolved"].resolve(); stamp(q["resolved"],t); return q,"inferred",time.perf_counter()-start

if not RUN_SWEEP:
    print("STATUS: RUN_SWEEP=False; no GPU inference or cleanup run.")
else:
    model=InstanSeg(MODEL_NAME,verbosity=1); rows=[]
    for t in SEED_THRESHOLDS:
        q,status,secs=run_one(model,t); assert compatible(q["resolved"],t); ids,pixels=labels(q["resolved"]); raw=attrs_summary(q["resolved"])
        clean=run_cleanup(q["resolved"],q["cleaned"],q["csv"],q["json"],q["overview"],SMALL_CROP,reference_channel_id=REFERENCE_CHANNEL_ID,native_shape=(CROP_H,CROP_W),chunk_size=2048,min_size=INSTANSEG_MIN_SIZE,reuse=CLEANUP_REUSE,overwrite=CLEANUP_OVERWRITE) if RUN_CLEANUP else None
        if clean is None: raise RuntimeError("RUN_CLEANUP must be True for this experiment")
        o,f=clean["original"],clean["final"]; rows.append({"seed_threshold":float(t),"run_status":status,"inference_minutes":None if secs is None else secs/60,"cleanup_minutes":clean.get("elapsed_minutes"),"raw_nuclei":raw["raw_nuclei"],"raw_cells":raw["raw_cells"],"proxy_cells":raw["proxy_cells"],"unmatched_nuclei":raw["unmatched_nuclei"],"raw_no_nucleus_cell_ids":len(ids[1]-ids[0]),"original_nuclei":o["nuclei"],"original_cells":o["cells"],"original_nuclear_pixels":o["nuclear_foreground_pixels"],"original_cell_pixels":o["cell_foreground_pixels"],"removed_nuclear_components":clean.get("removed_nuclear_components"),"removed_nuclear_pixels":clean.get("removed_nuclear_pixels"),"rejected_coordinated_ids":clean.get("rejected_coordinated_ids"),"removed_nucleus_free_cell_components":clean.get("removed_nucleus_free_cell_components"),"removed_nucleus_free_cell_pixels":clean.get("removed_nucleus_free_cell_pixels"),"rejected_unnucleated_ids":clean.get("rejected_unnucleated_ids"),"rejected_unnucleated_cell_pixels":clean.get("rejected_unnucleated_cell_pixels"),"final_coordinated_ids":f["nuclei"],"final_nuclear_pixels":f["nuclear_foreground_pixels"],"final_cell_pixels":f["cell_foreground_pixels"],"normalization_signature":raw["normalization_signature"],"resolved_zarr":str(q["resolved"]),"cleaned_zarr":str(q["cleaned"])})
    assert len({json.dumps(r["normalization_signature"],sort_keys=True) for r in rows})==1,"normalization bounds differ across thresholds"
    comparison=pd.DataFrame(rows).sort_values("seed_threshold").reset_index(drop=True); comparison.to_csv(SUMMARY_CSV,index=False)
    SUMMARY_JSON.write_text(json.dumps({"experiment":"seed_threshold_sweep_small_crop","source_crop":str(SMALL_CROP.resolve()),"crop_bounds_yx":list(CROP_BOUNDS_YX),"hotspot_source_bounds":list(HOTSPOT_SOURCE_BOUNDS),"fixed_native_bounds":list(FIXED_NATIVE_BOUNDS),"seed_thresholds":list(SEED_THRESHOLDS),"normalization_signature":rows[0]["normalization_signature"],"rows":rows},indent=2,sort_keys=True,default=str)+"\n")
    display(comparison.drop(columns=["normalization_signature"])); print({"summary_csv":str(SUMMARY_CSV),"summary_json":str(SUMMARY_JSON),"normalization_bounds_match":True})


## 3. Same-field multi-row visualization

Every threshold row uses the exact same fixed crop-local native field: DAPI, original labels, exact removed cells/nuclei in explicit blue/red RGBA, and cleaned labels. No scalar label colormap is used.


In [ ]:
def view(store,bounds):
    y0,y1,x0,x1=map(int,bounds); mh,mw=map(int,store.shape[-2:]); ys=np.clip(((2*np.arange(y0,y1)+1)*mh)//(2*CROP_H),0,mh-1); xs=np.clip(((2*np.arange(x0,x1)+1)*mw)//(2*CROP_W),0,mw-1); sy,sx=int(ys.min()),int(xs.min()); b=np.asarray(store[:,sy:int(ys.max())+1,sx:int(xs.max())+1]); return np.take(np.take(b,ys-sy,axis=1),xs-sx,axis=2)
def rgba(mask,color,alpha):
    z=np.zeros(mask.shape+(4,),dtype=np.uint8); z[mask,:3]=np.asarray(color,dtype=np.uint8); z[mask,3]=np.uint8(alpha); return z
with tifffile.TiffFile(str(SMALL_CROP)) as tif:
    ds=tif.series[0].aszarr(level=0); da=zarr.open(ds,mode="r"); dapi=np.asarray(da[REFERENCE_CHANNEL_ID,FIXED_NATIVE_BOUNDS[0]:FIXED_NATIVE_BOUNDS[1],FIXED_NATIVE_BOUNDS[2]:FIXED_NATIVE_BOUNDS[3]],dtype=np.float32); ds.close()
lo,hi=np.percentile(dapi,(1.0,99.8)); dapi=np.clip((dapi-lo)/max(float(hi-lo),1e-6),0,1)
if RUN_SWEEP and "comparison" in globals() and len(comparison)==len(SEED_THRESHOLDS):
    fig,ax=plt.subplots(len(comparison),4,figsize=(16,4*len(comparison)),squeeze=False)
    for r,row in enumerate(comparison.to_dict("records")):
        raw=zarr.open(row["resolved_zarr"],mode="r"); clean=zarr.open(row["cleaned_zarr"],mode="r"); rv,cv=view(raw,FIXED_NATIVE_BOUNDS),view(clean,FIXED_NATIVE_BOUNDS); rn=(rv[0]>0)&(cv[0]==0); rc=(rv[1]>0)&(cv[1]==0)
        ax[r,0].imshow(dapi,cmap="gray"); ax[r,0].set_title(f"seed={row['seed_threshold']:.2f} DAPI")
        ax[r,1].imshow(dapi,cmap="gray"); ax[r,1].imshow(rgba(rc,(0,82,220),145)); ax[r,1].imshow(rgba(rn,(235,35,35),230)); ax[r,1].set_title("removed: cells blue, nuclei red")
        ax[r,2].imshow(dapi,cmap="gray"); ax[r,2].contour(find_boundaries(rv[1],mode="outer"),[.5],colors=["#ffd21f"],linewidths=.45); ax[r,2].contour(find_boundaries(rv[0],mode="outer"),[.5],colors=["#20d9e8"],linewidths=.55); ax[r,2].set_title("original: cells yellow, nuclei cyan")
        ax[r,3].imshow(dapi,cmap="gray"); ax[r,3].contour(find_boundaries(cv[1],mode="outer"),[.5],colors=["#ffd21f"],linewidths=.45); ax[r,3].contour(find_boundaries(cv[0],mode="outer"),[.5],colors=["#20d9e8"],linewidths=.55); ax[r,3].set_title("after cleanup")
        for a in ax[r]: a.axis("off")
    fig.suptitle(f"fixed crop-local field y={FIXED_NATIVE_BOUNDS[0]}:{FIXED_NATIVE_BOUNDS[1]}, x={FIXED_NATIVE_BOUNDS[2]}:{FIXED_NATIVE_BOUNDS[3]}"); fig.tight_layout(rect=(0,0,1,.98)); fig.savefig(COMPARISON_PNG,dpi=180,bbox_inches="tight"); plt.show(); plt.close(fig)
    print({"comparison_png":str(COMPARISON_PNG),"hotspot_source_bounds":HOTSPOT_SOURCE_BOUNDS})
else: print("STATUS: visualization waits for RUN_SWEEP=True and five completed cleanups.")


## 4. Guardrails

The comparison reports descriptive segmentation burden, proxy/no-nucleus burden, cleanup removals, final coordinated IDs/pixels, and timing. It does not claim biological accuracy without ground truth.
